# Demo live — TimesFM-3 di fronte a uno shock

**Offline by design.** Questo notebook legge solo `results/*.parquet` — niente rete,
niente Colab, niente Hugging Face il giorno del talk. Se una cella sotto fallisce,
il fix è rilanciare gli esperimenti (`scripts/02..05`) e poi `06_make_figures.py`
per rigenerare `results/exp_mtg_demo_slice.parquet`, non aggiustare questo notebook.

Tutto il codice di rendering vive in `tfm3lab.plots` / `tfm3lab.figdata` — le stesse
funzioni che generano le figure delle slide, non una copia.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from tfm3lab import config, figdata, plots

plots.apply_style()

mtg_preds, mtg_source = figdata.load_mtg_predictions()
print(f"MTG predictions loaded from: {mtg_source.name}")

shock = pd.read_parquet(config.RESULTS_DIR / "exp_shock_raw_predictions.parquet")
lag = pd.read_parquet(config.RESULTS_DIR / "exp_shock_adaptation_lag.parquet")
shock_sp500 = shock[(shock["series"] == "SP500") & (shock["mode"] == "timesfm3_multivariate")]
print("Eventi disponibili:", sorted(shock_sp500["event"].unique()))

## Passo 0 — una carta, fino a un certo punto

`SERIES` e `ORIGIN` sono modificabili dal vivo: cambia carta o punto di taglio e
rilancia le due celle sotto. `ORIGIN=238` (2024-10-03) è scelto perché lontano dal
glitch TCGCSV del 2024-11-15 (vedi in fondo alla repo, `exp_mtg_data_glitch.png`) e
perché la finestra successiva inverte direzione in modo netto.

In [ ]:
SERIES = "The One Ring [LTR]"
ORIGIN = 238

mtg_truth = figdata.reconstruct_truth(mtg_preds)
sl = figdata.build_forecast_slice(mtg_preds, mtg_truth, SERIES, ORIGIN)

fig, ax = plt.subplots(figsize=(10, 5.5))
plots.plot_forecast_slice(sl, ax=ax, reveal=False)
plt.show()

## Passo 1 — cosa succede dopo? E cosa dice il modello?

TimesFM-3 vede solo la linea nera fino al taglio rosso. Rilancia la cella sotto per
la rivelazione: reale (nero), mediana del modello (blu), banda P10-P90 (blu chiaro),
naive — l'ultimo prezzo osservato, ripetuto (grigio tratteggiato).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
plots.plot_forecast_slice(sl, ax=ax, reveal=True)
plt.show()

## Passo 2 — lo stesso protocollo su un evento di mercato

Stesso concetto del grafico sopra, ora su SP500 attorno a uno shock: previsione **a un
giorno**, rifatta ogni mattina con tutto lo storico fino al giorno prima — non è una
previsione a 28 giorni come nel Passo 1. Il naive (grigio tratteggiato) è sovrapposto
apposta: guarda quanto il blu gli somiglia.

In [ ]:
EVENT_PRE = "Crollo Covid"

gap = figdata.naive_gap(shock_sp500, group_col="event")
row = gap[gap["event"] == EVENT_PRE].iloc[0]
print(
    f"{EVENT_PRE}: rapporto |previsione-naive|/|reale-naive| = {row['ratio']:.2f}, "
    f"corr(previsione, naive) = {row['corr']:.3f}"
)

sub_pre = shock_sp500[shock_sp500["event"] == EVENT_PRE]
fig, axes = plt.subplots(2, 1, figsize=(10, 6.5), sharex=True)
plots.plot_shock_reaction(sub_pre, axes=axes, title=EVENT_PRE)
plt.show()

## Passo 3 — stesso protocollo, evento post-cutoff

`EVENT_POST` modificabile dal vivo — confronta col Passo 2.

In [ ]:
EVENT_POST = "Shock dazi"

row = gap[gap["event"] == EVENT_POST].iloc[0]
print(f"{EVENT_POST}: rapporto = {row['ratio']:.2f}, corr = {row['corr']:.3f}")

sub_post = shock_sp500[shock_sp500["event"] == EVENT_POST]
fig, axes = plt.subplots(2, 1, figsize=(10, 6.5), sharex=True)
plots.plot_shock_reaction(sub_post, axes=axes, title=EVENT_POST)
plt.show()

## Adaptation lag — e perché guardarlo con sospetto

**Taglia questa sezione se il tempo stringe** — il messaggio di chiusura regge anche senza.

Il numero "pre-cutoff si riadatta in 1.7 giorni, post-cutoff in 9.5" nasconde che la
soglia di recupero è la mediana dell'errore **pre-evento** × 1.5, e quella mediana varia
7x fra eventi: la finestra pre-evento di Covid contiene già la rampa del crollo, quindi
supera la propria soglia quasi per costruzione. Il pannello sotto mostra la soglia dietro
ogni punto — guardalo prima di credere al numero sopra.

In [ ]:
detail = figdata.adaptation_lag_detail(shock, lag, series="SP500")
print(detail.to_string(index=False))

fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
plots.plot_adaptation_dots(detail, axes=axes)
plt.show()

## Messaggio di chiusura

> I foundation model per serie temporali non sono oracoli. Sono sistemi di aggiornamento
> probabilistico: riconoscono pattern osservati, ma uno shock veramente nuovo diventa
> prevedibile solo dopo che ha iniziato a lasciare una traccia nei dati — e prima di
> crederci, bisogna verificare che non lo stia semplicemente ricordando.